In [1]:
import torch
import gzip
import pickle
import requests
import os
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
from PIL import Image
import imageio.v2 as imageio
from IPython.display import Markdown, display, Video
from io import BytesIO

# Delving deeper into pytorch

The goal of this assesment is to delve deeper in the fundamentals of pytorch. Due to my background we will be using regresion models of the form $f : \mathbb{R} \rightarrow \mathbb{R}$ and start working with images. 

As we will see, many of the things we have been implementing so far, are wrapped in several ways in the pytorch library, for our easy usage. Pytorch is an excellent library with many ideas copied by Tensorflow or Jax. Perhaps one of their differential ideas is the `torch.nn module` (copied by Tensorflow and implemented in Flax, which is a Jax wrapper), and their data managment tools.

Pytorch has also copied many strenghts from Jax like forward automatic differentiation and being able of obtaining gradient computational graphs as functions. While Jax has many strenghts such as jit compilation (also copied by pytorch) or XLA compilation (which I think is also now in pytorch as well).

But we should be clear that one of the fathers of the automatic differentiation engine in pytorch, which comes from the Hips lab, was hired by google to create Jax.

## Datahandling

The nice thing about pytorch is that implements different modules to allow us nice transformations over different types of data. Note that this can be used within any machine model (including sklearn) because it is just a data processing pipeline.

In its beginning we had the torchvision module, but nowdays we also have torchaudio and torchtext.

Data handling in pytorch has two main components: a dataset and a dataloader.

A dataset is in charged of all the processes involved in a data handling machine learning pipeline. These include, for example:

* Loading the data from the disk into the RAM memory. If for example the data fits in memory you might decide to load all the data. If however the data is huge you can decide having pointers to memory, and load the data just on demand. You could also want to keep the paths to the different data, or just have a chunk of your data into memory, and later on load new data from disk.
* Applying transformations to the data: normalization, feature selection, data augmentation, etc. Many of these transformation are already implemented in pytorch, with depending on the module, will be more adequate for audio, video, images or text. Obviously, pytorch allows you to include your own transformations as function pointers.

A dataloader is in charged of "asking" the dataset for new data and give this data to use it so that you can use it in your machine learning pipeline. Cool things about dataloaders include:

* Multithreading dataloading: when data is expensive to preprocess, you can configure the dataloader to use parallel computing to perform these taks in parallel.
* You can also tell the dataloader how many data you want to receive from memory (for minibatch gradient descent) and so on.
* Since the dataloader "asks" the dataset, this obviously implies that the dataloader receives, as argument, the dataset you want it to operate on.

### Datasets

Let's work with a new dataset which is call de Mnist. Mnist consists of 60 thousand 28x28 pixel grayscale images. Each of these images correspond to a number from 0 to 9. 

One of the coolest things of torchvision is that it has many famous datasets included, and mnits is one of those. You can check, for example: `https://pytorch.org/vision/stable/datasets.html`.

However, since our goal is to understand how to construct our own dataset, let's do it. As you can see in the documentation: `https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset`, a dataset is a python class where we should overwrite two methods: `__getitem__()` and `__len__()`. The earlier is the way a dataloader will tell you which sample you need to provide back, while the later is the one telling the dataloader how many samples your dataset contains. 

One cool concept introduced in torchvision is the concept of transformations. Transformations are function pointers or classes implementing a `__call__` method, that can be used by our dataset to modify the image on the fly before giving it back through the get_item method. Note that in Machine Learning we might sometimes want to perform data preprocessing just once, but it would also be nice to perform data preprocessing on the fly, for some particular tasks.

When to use a function pointer and when to use a class? Well simple, when you want the transformation to be configurable.

Let's create our custom mnist classification dataset. To do so, we need to download the data to a specific directory, and then since this data fits in memory well, our dataset can directly keep the data in RAM memory.

Download your data from ```http://www.iro.umontreal.ca/~lisa/deep/data/mnist/mnist.pkl.gz``` and load it in memory. You will load a train, validation and test split. Join the three splits into a single split for the images and the labels. So in the end you should finish with two variables X, and T keeping images and labels.

In [2]:
dataset_dir = 'mnist.pkl.gz'

try:
    with gzip.open(dataset_dir, 'rb') as f:
        try:
            train_set, valid_set, test_set = pickle.load(f, encoding='latin1')
        except:
            train_set, valid_set, test_set = pickle.load(f)
except:
    raise ValueError("Download data from http://www.iro.umontreal.ca/~lisa/deep/data/mnist/mnist.pkl.gz and place it in a directory")
    
## Read and concatenate all data into X and Y 
X_tr, Y_tr = train_set
X_va, Y_va = valid_set
X_te, Y_te = test_set

X = np.vstack((X_tr,X_te,X_va))
T = np.hstack((Y_tr,Y_te,Y_va))

Now let's create the code for our dataset. Think in a way to apply transformations in the `__getitem__` method before returning the image. 

In [3]:
class MNISTDataset(torch.utils.data.Dataset):
    def __init__(self, X, T, transforms = None):
        ## call parent method
        super().__init__()
        self.X = X
        self.T = T
        self.transforms = transforms    
        
        
    def __len__(self):
        return len(self.T)
    
    def __getitem__(self, idx):
        X = self.X[idx]
        if self.transforms is not None:
            for T in self.transforms:
                X = T(X)
                
        return X, self.T[idx]

Once done, we can create our variable keeping our dataset.

Since this mnist version of the dataset is given prepared for a fully connected neural network, let's use a reshape transformation that turns our dataset into an image that we can visualize. Since mnist is 28 by 28 pixels we need to reshape into this shape. So when instancing our class passed into transforms the function pointer to the reshape function

In [4]:
def reshape(X):
    return np.reshape(X,(28,28))

mnist_dataset = MNISTDataset(X = X, T = T, transforms = reshape)

### Dataloader

To grab images from our dataset, we can use a dataloader, which has many usefull functions as I mentioned before. The coolest thing is that this dataloader provide us with an interable so that we can iterate over it to retrieve the full dataset. Once the full dataset has been retrieved, we can iterate again.

Let's use a batch_size of one, so that on each iteration we receive one image, and let's see the images


In [ ]:
mnist_loader = torch.utils.data.DataLoader(
                                                dataset = mnist_dataset,
                                                batch_size = 1, 
                                                shuffle=True,
                                                num_workers=1,
                                            )
fig, ax = plt.subplots(1,1)
video_filename = "aux4.mp4"
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

counter = 0
for x,y in mnist_loader:
    ax.clear()
    ax.imshow(x[0], cmap = 'gray')
    ax.set_title(f'Class {y.item()}')
    
    ## add frame for video creation
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  

    if counter == 10:
        break
        
    counter += 1

writer.close() 
plt.close()

In [1]:
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

NameError: name 'Video' is not defined

**Important:** In the past, we needed to apply a `transform.toTensor()` transformation in the dataset, to convert internal dataset type into torch tensors. However, it looks like this is now implicitly done by the data loader.

### Task 1: Implementing two custom transformations

Our goal will be to create different versions of the mnist dataset that we will use to train convolutional models and to learn models using self-supervision, which is the fundamental machine learning technique used by language models nowdays (such as your favourite tool chatGPT).

We will create a transformation to randomly rotate images a given angle, and a transformation to randomly mask the image using a black square.

First of all, you must think if you want this transformation configurable or not. If you want the configurable (for example the size of the mask can vary or the maximum angle of rotation) you will need to create a python instance class.

I'll make both of them configurable.

In [ ]:
def reshape(X):
    return np.reshape(X,(28,28))

class RandomBlankSquare:
    def __init__(self,min_h,max_h, min_v,max_v):
        self.min_h = min_h
        self.max_h = max_h
        self.min_v = min_v
        self.max_v = max_v
        
    def __call__(self, x):
        min_h_idx = np.random.randint(0,self.min_h, size=(1,), dtype = int).item()
        max_h_idx = np.random.randint(self.min_h,self.max_h, size=(1,), dtype = int).item()

        min_v_idx = np.random.randint(0,self.min_v, size=(1,), dtype = int).item()
        max_v_idx = np.random.randint(self.min_v,self.max_v, size=(1,), dtype = int).item()

        x[min_v_idx:max_v_idx,min_h_idx:max_h_idx] = 0.0

        return x

class RandomRotation:
    def __init__(self,max_degree):
        self.max_deg = max_degree
        
    def __call__(self, x):
        rot_deg = np.random.random(size=(1,)).item()**2*self.max_deg - self.max_deg 
        
        x = Image.fromarray(x)
        x = x.rotate(rot_deg)
        x = np.array(x)
        return x

Now experiment in using these transformations. What happens if you do not reshape first?

In [ ]:
mnist_dataset = MNISTDataset(
                                X = X, 
                                T = T, 
                                transforms = [
                                            reshape, 
                                            RandomBlankSquare(min_h = 10, max_h = 20,min_v = 10, max_v = 20),
                                            RandomRotation(70),
                                        ]
                         )


mnist_loader = torch.utils.data.DataLoader(
                                                dataset = mnist_dataset,
                                                batch_size = 1, 
                                                shuffle=True,
                                                num_workers=1,
                                            )
video_filename = "/tmp/aux4.mp4"
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")
fig, ax = plt.subplots(1,1)

counter = 0
for x,y in mnist_loader:
    
    ax.imshow(x[0], cmap = 'gray')
    ax.set_title(f'Class {y.item()}')
    
    ## add frame for video creation
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  
    
    if counter == 10:
        break

    counter += 1

writer.close()
plt.close()

In [ ]:
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

### Task 2: Creating a pytorch dataset with our regresions problems.

Remember from our previous task we had the following regresion data.

* task $f: \mathbb{R} \rightarrow \mathbb{R}$
$$
(x_1,t_1) = (0,0.2)\\
(x_2,t_2) = (1,0.5)\\
(x_3,t_3) = (2,2.8)
$$

* task $f: \mathbb{R} \rightarrow [0,1]$
$$
\begin{split}
(x_1,t_1) &= (-0.13459237,0)\\
(x_2,t_2) &= (-3.3015387,0)\\
(x_3,t_3) &= (0.74481176,0)\\
(x_4,t_4) &= (2.62434536,1)\\
(x_5,t_5) &= (0.38824359,1)\\
(x_6,t_6) &= (0.47182825,1)\\
(x_7,t_7) &= (-0.07296862,1)\\
\end{split}
$$


* task $f: \mathbb{R}^2 \rightarrow [0,1]$
$$
\begin{split}
(x^1_{1},x^1_{2},t^1) &= (0,1,0)\\
(x^2_{1},x^2_{2},t^2) &= (1.5,2.0,0)\\
(x^3_{1},x^3_{2},t^3) &= (2,1,0)\\
(x^4_{1},x^4_{2},t^4) &= (5,3,0)\\
(x^5_{1},x^5_{2},t^5) &= (3,4,1)\\
(x^6_{1},x^6_{2},t^6) &= (4,5,1)\\
(x^7_{1},x^7_{2},t^7) &= (5,1,1)\\
\end{split}
$$

Create the three datasets containing one of these datasets and wrap them with their corresponding loaders.

In [ ]:
class RtoR(torch.utils.data.Dataset):
    def __init__(self):
        ## call parent method
        super().__init__()
        self.X = np.array([0,1,2], dtype = np.float32).reshape(3,1)
        self.T = np.array([0.2,0.5,2.8], dtype = np.float32).reshape(3,1)
        
    def __len__(self):
        return self.T
    
    def __getitem__(self, idx):
        return self.X[idx], self.T[idx]

class Rto01(torch.utils.data.Dataset):
    def __init__(self):
        ## call parent method
        super().__init__()
        self.X = np.array([-0.13459237,-3.3015387,0.74481176,2.62434536,0.38824359,0.47182825,-0.07296862], dtype = np.float32).reshape(7,1)
        self.T = np.array([0,0,0,1,1,1,1], dtype = np.float32).reshape(7,1)
    
    def __len__(self):
        return self.T
    
    def __getitem__(self, idx):
        return self.X[idx], self.T[idx]

class R2to01NL(torch.utils.data.Dataset):
    def __init__(self):
        ## call parent method
        super().__init__()

        # input to our model. Represents time in seconds
        self.X = np.array([[0,1],
                           [1.5,2.0],
                           [2,1],
                           [5,3],
                           [3,4],
                           [4,5],
                           [5,1]], dtype = np.float32).reshape(7,2)

        # outputs associated to each input. 
        self.T =  np.array([0,0,0,0,1,1,1], dtype = np.float32).reshape(7,1)

    def __len__(self):
        return self.T
    
    def __getitem__(self, idx):
        return self.X[idx], self.T[idx]
        
rtor_loader = torch.utils.data.DataLoader(
                                            dataset = RtoR(),
                                            batch_size = len(RtoR()), 
                                            shuffle=True,
                                            num_workers=1,
                                        )

rto01_loader = torch.utils.data.DataLoader(
                                            dataset = Rto01(),
                                            batch_size = len(Rto01()), 
                                            shuffle=True,
                                            num_workers=1,
                                         )

r2to01nl_loader = torch.utils.data.DataLoader(
                                            dataset = R2to01NL(),
                                            batch_size = len(R2to01NL()), 
                                            shuffle=True,
                                            num_workers=1,
                                         )

## Torch nn module

The torch nn module is perhaps one of the greatest modules I have used as a machine learning researcher. Why? Basically because you have 100% flexibility in what you want to do and how to do it, and the torch.nn module handles the boring parts, like tracking which parameters do you want to optimize, moving your model parameters to the GPU or switching the behaviour of different classes.

So the goal of the torch nn module is making the creation of your machine learning models easier. Again, this is Neural Network's agnostic; you can use the torch nn module to implement any machine learning model you want.

Like with dataloaders, the torch nn module has a base class, call module, that any model must inherit. In this case we are required to overwrite the `forward` method, and as you'll see while not mandatory the `__init__` one.

We will start seeing the different beautiful features about this module during our lectures.

So for analogy with what you have done with sklearn, let's create a linear model in pytorch, that can be adapted two different classes.

In this case, the basic structure will be something like:

```python
import torch.nn as nn
import torch.nn.functional as F

class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        pass
    def forward(self,x):
        pass
```

#### `__init__` method

The goal of these method is to initialize the computational graph that will be implementing your desired machine learning model. As you know, machine learning is about defining a set of computer operations that implement a function, and adjusting these operations for our task at hand. This adjustment is done via its parameters.

#### `forward` method

The forward method is in charged of using the parameters to implement the set of operations that implements your desired function / computational graph.


In [ ]:
class LinearModel(nn.Module):
    def __init__(self, dim_in, dim_out, link_function, loss_function):
        super().__init__()
        
        ## create parameters
        self.W = torch.tensor(np.random.randn(dim_in,dim_out), requires_grad = True, dtype = torch.float32)
        self.b = torch.zeros(dim_out, requires_grad = True, dtype = torch.float32)
        
        ## trace parameters
        self.parameter_list = []
        self._trace_parameter(self.W)
        self._trace_parameter(self.b)
        
        ## link and loss funciton
        self.link = link_function
        self.loss = loss_function
        
    def forward(self,x, apply_link):
        y = x @ self.W + self.b
    
        if apply_link:
            y = self.link(y)
        
        return y
    
    def compute_loss(self,t,y):
        return self.loss(y,t)
    
    def _trace_parameter(self,p):
        self.parameter_list.append(p)
    
    def get_parameters(self):
        return self.parameter_list

Finally, let's create a model instance:

In [ ]:
model = LinearModel(dim_in = 1, dim_out = 1, link_function = torch.sigmoid, loss_function = nn.MSELoss())

Let's now train our model

In [ ]:
## ================= ##
## Training pipeline ##
## ================= ##
## to display learning functions
video_filename = "/tmp/aux4.mp4"
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

fig, (ax1, ax2) = plt.subplots(1,2)
N_points = 100
x_range = torch.linspace(-4,4,N_points).reshape(N_points,1)
loss_acc_list = []

## create your data loader
ror_loader = torch.utils.data.DataLoader(
                                            dataset = RtoR(),
                                            batch_size = len(RtoR()), 
                                            shuffle=True,
                                            num_workers=1,
                                        )

## Model Initialization
def linear(x):
    '''Linear link function.'''
model = LinearModel(dim_in = 1, dim_out = 1, link_function = linear, loss_function = nn.MSEloss() )

## optimization hyperparaemteres
epochs = 100
lr = 0.1
parameters = model.get_parameters()

## training loop
for e in range(epochs):
    loss_acc = 0.0
    for x,t in ror_loader:
        
        ## forward
        y = model(x, apply_link = False)
        L = model.compute_loss(t,y)
        loss_acc += L.item()
        
        ## backward
        L.backward()
        
        ## update
        for p in parameters:
            p.data = p.data -lr*p.grad
            
        ## zero grad
        for p in parameters:
            p.grad.zero_()
       
    print(f"On epoch {e} got loss {loss_acc}", end = "\r")
    
    ## plot regressed line and loss
    with torch.no_grad():
        ## add loss to loss list
        loss_acc_list.append(loss_acc)
        
        ax1.cla()
        ax2.cla()
        
        ax1.plot(x,t,'+')
        ax1.plot(x_range, model(x_range, apply_link = False))
        ax1.set_ylim([0,3])
        
        ax2.plot(np.arange(e+1), loss_acc_list)
        ax2.set_xlim([0,epochs])
    
        ## add frame for video creation
        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
        
        buf.seek(0)
        frame = imageio.imread(buf) 
        writer.append_data(frame) 

writer.close()
plt.close()

display(Video(data=video_filename, embed=True))
os.remove(video_filename)

In [ ]:
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

However, this might be a good solution for an initial/toy application. Note that once you create a more complex application, you would probably require additional methods that reset this parameter list or that are able to register parameters coming from a class implementing complex functionalities that your model wants to use. Obviously, the nn module is able to handle the way you do this through different functionalities implemented. In this tutorial, we will cover 9 of them: 
* **`def parameters()`**
* **`def named_parameters()`**
* **`nn.Parameter()`**
* **`def to()`**
* **`def zero_grad()`**
* `def train() def eval()`
* `nn.Sequential`
* `nn.ParameterList`
* `nn.ModuleList`

First, suppose you want to implement a machine learning operation that you want to use within a bigger model (this is very typical, for example, when implementing a residual network or a transformer model). We can create this operation as a `nn.Module`. Let's create a Linear layer.

In [ ]:
class Linear(...):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        ## create parameters
        self.W = nn.Parameter(torch.tensor(np.random.randn(...), dtype = torch.float32))
        self.b = nn.Parameter(torch.zeros(..., dtype = torch.float32))
    def forward(self, x):
        return ...

Let's now create this linear model using our created Linear module.